# Create data splits for training the nnU-Net
The following notebook creates the data splits used to train a nnU-Net model via 5 fold cross-validation. We make sure that one patient cannot be part of the train and validation split at the same time. Furthermore, we stratify based on whether the data originates from the HQcolon dataset (mostly non-collapsed cases) or our semi-automatic method (mostly collapsed cases). 

In [2]:
# import libraries
import json
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [3]:
# load metadata
metadata_tcia = pd.read_excel("../../../data/raw/metadata/metadata_tcia.xlsx")
metadata_ku = pd.read_json("../../../data/raw/metadata/metadata_ku.json", lines=True)

# get Patient Sex count of data
group_by_patient = metadata_tcia.groupby("Patient ID").first().reset_index()
sex_counts = group_by_patient["Patient Sex"].value_counts()
print(sex_counts)

# merge dataframes by "Study Instance UID" and "InstanceUID"
merged_data = pd.merge(metadata_tcia, metadata_ku, left_on="Series Instance UID", right_on="InstanceUID", how="inner")
# Display the first few rows of the merged data
print(merged_data.shape)
merged_data.head(2)


Patient Sex
F    401
M    353
U     19
O     12
Name: count, dtype: int64
(1714, 61)


,Patient ID,Patient Name,Patient Birth Date,Patient Sex,Ethnic Group,Phantom,Species Code,Species Description,Study Instance UID,Study Date,...,patients_age,slice_location,gender,new_sub_id,scan,position,mha_path,dicom_path,split,segmentation_path
0,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,59.0,551.9,F,sub001,1,prone,converted/sub001/sub001_pos-prone_scan-1_conv-...,raw/sub001/sub001_pos-prone_scan-1.zip,NaN,NaN
1,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,59.0,562.2,F,sub001,1,supine,converted/sub001/sub001_pos-supine_scan-1_conv...,raw/sub001/sub001_pos-supine_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...


### Filter data based on patient position and scanner

In [4]:
# only keep ffs, ffp, and hfs, hfp
def filter_position(position):
    if position in ["FFS", "FFP", "HFS", "HFP"]:
        return position
    else:
        return None
merged_data["patient_position"] = merged_data["patient_position"].apply(filter_position)
# drop rows with None values in patient_position
merged_data = merged_data[merged_data["patient_position"].notnull()]

print(merged_data.shape)

(1666, 61)


In [5]:
# build base_key from merged_data using new_sub_id + position
merged_data["sub_num"] = merged_data["new_sub_id"].str.extract(r"(\d+)").astype(int)
merged_data["merged_base_key"] = (
    "colon_" + merged_data["sub_num"].apply(lambda x: f"{x:04d}") + "-" + merged_data["position"]
)
merged_data

,Patient ID,Patient Name,Patient Birth Date,Patient Sex,Ethnic Group,Phantom,Species Code,Species Description,Study Instance UID,Study Date,...,gender,new_sub_id,scan,position,mha_path,dicom_path,split,segmentation_path,sub_num,merged_base_key
0,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,F,sub001,1,prone,converted/sub001/sub001_pos-prone_scan-1_conv-...,raw/sub001/sub001_pos-prone_scan-1.zip,NaN,NaN,1,colon_0001-prone
1,1.3.6.1.4.1.9328.50.4.0001,1.3.6.1.4.1.9328.50.4.0001,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1,2000-01-01 00:00:00.0,...,F,sub001,1,supine,converted/sub001/sub001_pos-supine_scan-1_conv...,raw/sub001/sub001_pos-supine_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...,1,colon_0001-supine
2,1.3.6.1.4.1.9328.50.4.0002,1.3.6.1.4.1.9328.50.4.0002,NaN,M,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1240,2000-01-01 00:00:00.0,...,M,sub002,1,prone,converted/sub002/sub002_pos-prone_scan-1_conv-...,raw/sub002/sub002_pos-prone_scan-1.zip,NaN,NaN,2,colon_0002-prone
3,1.3.6.1.4.1.9328.50.4.0002,1.3.6.1.4.1.9328.50.4.0002,NaN,M,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.1240,2000-01-01 00:00:00.0,...,M,sub002,1,supine,converted/sub002/sub002_pos-supine_scan-1_conv...,raw/sub002/sub002_pos-supine_scan-1.zip,NaN,NaN,2,colon_0002-supine
5,1.3.6.1.4.1.9328.50.4.0003,1.3.6.1.4.1.9328.50.4.0003,NaN,M,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.2319,2000-01-01 00:00:00.0,...,M,sub003,1,supine,converted/sub003/sub003_pos-supine_scan-1_conv...,raw/sub003/sub003_pos-supine_scan-1.zip,train,segmentations/segmentations-regionalgrowing-qc...,3,colon_0003-supine
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1709,1.3.6.1.4.1.9328.50.4.0784,1.3.6.1.4.1.9328.50.4.0784,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.892718,2000-01-01 00:00:00.0,...,F,sub823,1,prone,converted/sub823/sub823_pos-prone_scan-1_conv-...,raw/sub823/sub823_pos-prone_scan-1.zip,NaN,NaN,823,colon_0823-prone
1710,1.3.6.1.4.1.9328.50.4.0785,1.3.6.1.4.1.9328.50.4.0785,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.893841,2000-01-01 00:00:00.0,...,F,sub824,1,prone,converted/sub824/sub824_pos-prone_scan-1_conv-...,raw/sub824/sub824_pos-prone_scan-1.zip,NaN,NaN,824,colon_0824-prone
1711,1.3.6.1.4.1.9328.50.4.0785,1.3.6.1.4.1.9328.50.4.0785,NaN,F,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.893841,2000-01-01 00:00:00.0,...,F,sub824,1,supine,converted/sub824/sub824_pos-supine_scan-1_conv...,raw/sub824/sub824_pos-supine_scan-1.zip,NaN,NaN,824,colon_0824-supine
1712,1.3.6.1.4.1.9328.50.4.0786,1.3.6.1.4.1.9328.50.4.0786,NaN,M,NaN,NO,337915000,Homo sapiens,1.3.6.1.4.1.9328.50.4.894940,2000-01-01 00:00:00.0,...,M,sub825,1,prone,converted/sub825/sub825_pos-prone_scan-1_conv-...,raw/sub825/sub825_pos-prone_scan-1.zip,test,segmentations/segmentations-regionalgrowing-qc...,825,colon_0825-prone


### Create held-out test set

In [6]:
# keep only Philips and Toshiba
held_out = merged_data[merged_data["Manufacturer"].isin(["Philips", "TOSHIBA"])]

# remove manufacturer Philips or Toshiba
merged_data = merged_data[~merged_data["Manufacturer"].isin(["Philips", "TOSHIBA"])]
print(merged_data.shape)


(1552, 63)


In [ ]:
# write held-out base keys to .txt
with open("../../../data/processed/metadata/base_keys_held_out_data.txt", "w") as f:
    for key in held_out["merged_base_key"]:
        f.write(f"{key}\n")

### Create splits for cross-validation

In [8]:
# read txt files
collapsed_files = pd.read_csv("../../../data/interim/filenames_per_origin/filenames_masks_semi_automatic.txt", header=None, names=["file_path"])
non_collapsed_files = pd.read_csv("../../../data/interim/filenames_per_origin/filenames_masks_non_collapsed.txt", header=None, names=["file_path"])
raw_files = pd.read_csv("../../../data/interim/filenames_per_origin/filenames_all_raw_scans.txt", header=None, names=["file_path"])

In [ ]:
# extract base keys from masks
collapsed_keys = collapsed_files["file_path"].str.replace(".mha", "", regex=False)
non_collapsed_keys = non_collapsed_files["file_path"].str.replace(".mha", "", regex=False)
all_mask_keys = set(collapsed_keys) | set(non_collapsed_keys)

# extract base key from raw images (remove _0000 suffix)
raw_files["base_key"] = raw_files["file_path"].str.replace("_0000.mha", "", regex=False)

# keep only images that have a corresponding mask
raw_with_mask = raw_files[raw_files["base_key"].isin(all_mask_keys)].copy()

# remove files that are not in merged_data
raw_with_mask = raw_with_mask[raw_with_mask["base_key"].isin(merged_data["merged_base_key"])].copy()

In [10]:
# extract patient ID (e.g. "colon_0001" from "colon_0001-prone")
raw_with_mask["patient_id"] = raw_with_mask["base_key"].str.extract(r"(colon_\d+)")

collapsed_set = set(collapsed_keys)
non_collapsed_set = set(non_collapsed_keys)

# assign a collapse label per scan
def get_collapse_label(key):
    in_c = key in collapsed_set
    in_nc = key in non_collapsed_set
    if in_c:
        return "collapsed"
    elif in_nc:
        return "non_collapsed"
    else:
        print(key)

raw_with_mask["collapse_label"] = raw_with_mask["base_key"].apply(get_collapse_label)

print("Overall collapse label distribution:")
print(raw_with_mask["collapse_label"].value_counts())
print()

# 5-fold cross-validation grouped by patient, stratified by collapse label
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

splits = []
for fold, (train_idx, val_idx) in enumerate(sgkf.split(
    raw_with_mask,
    y=raw_with_mask["collapse_label"],
    groups=raw_with_mask["patient_id"]
)):
    train_files = raw_with_mask.iloc[train_idx]["file_path"].tolist()
    val_files   = raw_with_mask.iloc[val_idx]["file_path"].tolist()
    splits.append({"fold": fold, "train": train_files, "val": val_files})

    train_labels = raw_with_mask.iloc[train_idx]["collapse_label"].value_counts().to_dict()
    val_labels   = raw_with_mask.iloc[val_idx]["collapse_label"].value_counts().to_dict()
    print(f"Fold {fold}: train={len(train_files)} {train_labels}, "
          f"val={len(val_files)} {val_labels}")

Overall collapse label distribution:
collapse_label
non_collapsed    380
collapsed        307
Name: count, dtype: int64

Fold 0: train=550 {'non_collapsed': 304, 'collapsed': 246}, val=137 {'non_collapsed': 76, 'collapsed': 61}
Fold 1: train=550 {'non_collapsed': 304, 'collapsed': 246}, val=137 {'non_collapsed': 76, 'collapsed': 61}
Fold 2: train=550 {'non_collapsed': 304, 'collapsed': 246}, val=137 {'non_collapsed': 76, 'collapsed': 61}
Fold 3: train=548 {'non_collapsed': 304, 'collapsed': 244}, val=139 {'non_collapsed': 76, 'collapsed': 63}
Fold 4: train=550 {'non_collapsed': 304, 'collapsed': 246}, val=137 {'non_collapsed': 76, 'collapsed': 61}


In [ ]:
# write splits to JSON file for nnUNet
nnunet_splits = [
    {
        "train": [f.replace("_0000.mha", "") for f in split["train"]],
        "val":   [f.replace("_0000.mha", "") for f in split["val"]],
    }
    for split in splits
]

with open("../../nnunet/preprocessed/Dataset999_Colon/splits_final.json", "w") as f:
    json.dump(nnunet_splits, f, indent=2)
